# Estimación del parámetro *b* de Gutenberg–Richter: Máxima Verosimilitud vs Bayes Empírico
**Grupo 7:** Benjamín Iribarren, Ramón Espinoza, Gaspar Albornoz · Estadística Avanzada (ICD3152)

**Idea central (se mantiene):** estimar *b* por macrozona con MLE y compararlo con Bayes Empírico (shrinkage hacia la media global), validando con bootstrap, jackknife, bloques por año y submuestreo.

**Qué cambió respecto de la versión anterior:** (1) ahora se filtra por $M_c=4{,}2$ *antes* de estimar; (2) MLE con corrección de Utsu ($\Delta m=0{,}1$); (3) bootstrap con el mismo estimador y el mismo $M_c$; (4) MICE con 5 imputaciones; (5) Bayes Empírico normal–normal con DerSimonian–Laird (μ̂ ponderado); (6) se agregan jackknife por año, bootstrap por bloques, KS, sensibilidad ($M_c$, MWconv, Perú/Bolivia) y submuestreo; (7) se retira la afirmación de proceso de Poisson, porque el CV de los tiempos entre eventos es ≫ 1.

## 0. Configuración y carga de datos

In [ ]:
import numpy as np, pandas as pd, json, matplotlib.pyplot as plt
from scipy import stats
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
plt.style.use('seaborn-v0_8-whitegrid')

SEED, MC, DM, B_BOOT, R_SUB = 42, 4.2, 0.1, 1000, 500   # configuración reproducible
LOG10E = np.log10(np.e)
rng = np.random.default_rng(SEED)

df = pd.read_csv('eventos_filtrados_mapa.csv')
df['datetime'] = pd.to_datetime(df['datetime'], errors='coerce')
df['anio'] = df['datetime'].dt.year
print(df.shape, '| rango:', df.anio.min(), '-', df.anio.max())
print('Faltantes por columna:'); print(df.isna().sum()[lambda s: s>0])

## 1. Imputación (MICE) y macrozonas
Solo `depth` tiene faltantes (293, 0,66 %). Se hacen **5 imputaciones iterativas con muestreo posterior** y se promedian, exigiendo profundidad ≥ 0. La profundidad es descriptiva: **no entra a la estimación de b**.

In [ ]:
cols = ['lon','lat','mag','depth']
imps = [IterativeImputer(max_iter=10, sample_posterior=True, random_state=SEED+i).fit_transform(df[cols])[:,3] for i in range(5)]
mask = df['depth'].isna()
df['depth_imp'] = df['depth']
df.loc[mask,'depth_imp'] = np.clip(np.mean(imps,axis=0),0,None)[mask.values]
print('Imputados:', mask.sum(), '| profundidad imputada media: %.1f km' % df.loc[mask,'depth_imp'].mean())

ZN = ['Norte Grande','Norte Chico','Zona Central','Zona Sur','Zona Austral']
df['z'] = np.select([df.lat>=-26, df.lat>=-32, df.lat>=-38, df.lat>=-43],[0,1,2,3],4)
df['MacroZ'] = df.z.map(dict(enumerate(ZN)))
fuera = (df.lat >= -18.35)
print('Eventos con lat >= -18.35 (Perú/Bolivia): %d (%.1f %%), todos caen en Norte Grande' % (fuera.sum(), 100*fuera.mean()))
print(df.MacroZ.value_counts().reindex(ZN))

## 2. Exploración: comparabilidad del catálogo

In [ ]:
print('Tipos de magnitud:', df.mag_type.value_counts().to_dict())
d = df.sort_values('datetime')
dt = d.datetime.diff().dt.total_seconds().div(3600).dropna(); dt = dt[dt>0]
print('CV de los tiempos entre eventos: %.1f  (un proceso de Poisson daría CV = 1)' % (dt.std()/dt.mean()))
fig,ax = plt.subplots(1,2,figsize=(13,4))
yc = df[df.anio>=1960].groupby('anio').size(); ax[0].bar(yc.index, yc.values); ax[0].set_title('Eventos por año (1960–2025)')
pd.crosstab(pd.cut(df.anio,[0,1959,1989,2004,2025],labels=['<1960','1960-89','1990-2004','2005-25']), df.mag_type, normalize='index').plot.bar(stacked=True, ax=ax[1]); ax[1].set_title('Tipo de magnitud por período')
plt.tight_layout(); plt.show()

**Lectura:** el registro crece mucho en décadas recientes y la mezcla de escalas (Mw, mb, Md, ML) cambia por período. Además el CV ≫ 1 indica *dependencia temporal* (réplicas): los eventos no son independientes, y eso se tratará con bootstrap por bloques de años.

## 3. Magnitud de completitud $M_c$
Máxima curvatura (moda del histograma con $\Delta m=0{,}1$) más una corrección operativa de +0,2.

In [ ]:
h, e = np.histogram(df.mag, bins=np.arange(3.65, 9.65, DM))
mc_maxc = round(e[np.argmax(h)] + DM/2, 1)
print('Mc por máxima curvatura:', mc_maxc, '| Mc adoptada (+0,2):', MC)
d = df[df.mag >= MC].copy()
print('Eventos con M >= Mc: %d (%.1f %% del catálogo)' % (len(d), 100*len(d)/len(df)))
plt.figure(figsize=(8,4)); plt.hist(df.mag, bins=np.arange(3.65,8.05,DM), color='lightgray', edgecolor='k')
plt.axvline(mc_maxc, color='C1', ls='--', label='Mc MaxC'); plt.axvline(MC, color='g', label='Mc adoptada'); plt.legend(); plt.title('Histograma de magnitudes y Mc'); plt.show()

## 4. MLE de $b$ con corrección de Utsu
Con $Y=M-(M_c-\Delta m/2)\sim\text{Exp}(\beta)$, $\beta=b\ln 10$: $\ell(\beta)=n\ln\beta-\beta\sum Y_i$, $\hat\beta=n/\sum Y_i$, luego
$$\hat b=\frac{\log_{10}e}{\bar M-(M_c-\Delta m/2)},\qquad SE(\hat b)\approx \hat b/\sqrt n.$$

In [ ]:
def b_mle(m, mc=MC, dm=DM):
    return LOG10E / (np.mean(m) - (mc - dm/2))

R = pd.DataFrame({'zona':ZN})
R['n']  = [int((d.z==k).sum()) for k in range(5)]
R['b']  = [b_mle(d.mag[d.z==k]) for k in range(5)]
R['se'] = R.b/np.sqrt(R.n)
# Bootstrap iid (eventos), IC percentil 95 %
lo, hi = [], []
for k in range(5):
    m = d.mag[d.z==k].values
    bs = [b_mle(rng.choice(m, len(m))) for _ in range(B_BOOT)]
    lo.append(np.percentile(bs,2.5)); hi.append(np.percentile(bs,97.5))
R['ic_lo'], R['ic_hi'] = lo, hi
R.round(4)

In [ ]:
plt.figure(figsize=(8,4))
plt.errorbar(R.b, range(5), xerr=[R.b-R.ic_lo, R.ic_hi-R.b], fmt='o', capsize=4)
plt.yticks(range(5), ZN); plt.gca().invert_yaxis(); plt.xlabel('b'); plt.title('b por macrozona (IC bootstrap iid 95 %)'); plt.show()

# Ajuste visual de la ley en una zona
k=0; m=d.mag[d.z==k]; ms=np.arange(MC, 7.6, 0.1); N=[(m>=x-1e-9).sum() for x in ms]
plt.figure(figsize=(7,4)); plt.semilogy(ms,N,'o'); plt.semilogy(ms, N[0]*10**(-R.b[k]*(ms-MC)),'r'); plt.title('Gutenberg–Richter, '+ZN[k]); plt.show()

## 5. Bayes Empírico (normal–normal, DerSimonian–Laird)
$\hat b_i\mid\theta_i\sim N(\theta_i,\sigma_i^2)$, $\theta_i\sim N(\mu,\tau^2)$. Con pesos $w_i=1/\sigma_i^2$:
$\hat\mu=\sum w_i\hat b_i/\sum w_i$, $Q=\sum w_i(\hat b_i-\hat\mu)^2$, $\hat\tau^2=\max\{0,(Q-(k-1))/(\sum w_i-\sum w_i^2/\sum w_i)\}$,
$B_i=\sigma_i^2/(\sigma_i^2+\hat\tau^2)$, $\hat\theta_i=\hat\mu+(1-B_i)(\hat b_i-\hat\mu)$.
Nota: el $\hat\mu$ de DL es **ponderado** (≈0,855), no el promedio simple (≈0,775) usado en la versión anterior de la presentación.

In [ ]:
def eb_dl(b, s2):
    b, s2 = np.asarray(b), np.asarray(s2); w = 1/s2
    mu = (w*b).sum()/w.sum(); Q = (w*(b-mu)**2).sum()
    c = w.sum() - (w**2).sum()/w.sum(); tau2 = max(0, (Q-(len(b)-1))/c)
    Bi = s2/(s2+tau2)
    return dict(mu=mu, Q=Q, p=stats.chi2.sf(Q,len(b)-1), tau=np.sqrt(tau2), B=Bi, theta=mu+(1-Bi)*(b-mu))

E = eb_dl(R.b, R.se**2)
R['b_EB'] = E['theta']; R['contraccion_%'] = 100*E['B']
print('mu = %.3f | tau = %.3f | Cochran Q = %.1f (p = %.2e)' % (E['mu'],E['tau'],E['Q'],E['p']))
R[['zona','n','b','b_EB','contraccion_%']].round(4)

In [ ]:
plt.figure(figsize=(8,4)); plt.scatter(R.b, range(5), label='MLE'); plt.scatter(R.b_EB, range(5), marker='D', label='Bayes Empírico')
plt.axvline(E['mu'], ls='--', c='gray', label='media global'); plt.yticks(range(5), ZN); plt.gca().invert_yaxis(); plt.legend(); plt.xlabel('b'); plt.show()

**Lectura:** como $\hat\tau\gg$ SE típico, la contracción es pequeña (0,4 %–5,4 %) y MLE ≈ EB con estas macrozonas grandes y heterogéneas.

## 6. Validación: jackknife por año, bootstrap por bloques y ajuste distribucional

In [ ]:
jk_se, blk_se = [], []
for k in range(5):
    g = d[d.z==k].groupby('anio').mag.agg(['sum','count']); S, Cn = g['sum'].values, g['count'].values
    G = len(S)
    # Jackknife leave-one-year-out
    bj = np.array([LOG10E/((S.sum()-S[i])/(Cn.sum()-Cn[i]) - (MC-DM/2)) for i in range(G)])
    jk_se.append(np.sqrt((G-1)/G*((bj-bj.mean())**2).sum()))
    # Bootstrap por bloques: remuestrear años completos
    bb = []
    for _ in range(B_BOOT):
        i = rng.integers(0,G,G); bb.append(LOG10E/(S[i].sum()/Cn[i].sum() - (MC-DM/2)))
    blk_se.append(np.std(bb, ddof=1))
R['se_jackknife'], R['se_bloques'] = jk_se, blk_se
R['razon_bloques/analitico'] = R.se_bloques/R.se

# Heterogeneidad y contracción usando el SE por bloques (más realista ante dependencia)
Eb = eb_dl(R.b, R.se_bloques**2)
print('Con SE por bloques: Q = %.1f (p = %.2e) | contracción EB %.1f %% – %.1f %%' % (Eb['Q'],Eb['p'],100*Eb['B'].min(),100*Eb['B'].max()))

# KS sobre excesos; las magnitudes están redondeadas a 0,1, por eso se agrega ruido uniforme ±Δm/2
ks=[]
for k in range(5):
    y = d.mag[d.z==k].values + rng.uniform(-DM/2,DM/2,(d.z==k).sum()) - (MC-DM/2)
    ks.append(stats.kstest(y,'expon',args=(0,y.mean())).statistic)
R['KS_D']=ks
R[['zona','se','se_jackknife','se_bloques','razon_bloques/analitico','KS_D']].round(4)

**Lectura:** el SE por bloques es varias veces mayor que el analítico (dependencia temporal). El KS de la Zona Austral (≈0,10 frente a ≈0,03 del resto) es una alerta: el modelo exponencial ajusta peor ahí. Los p-valores del KS no se reportan porque los parámetros se estiman de los mismos datos.

## 7. Robustez: umbral $M_c$, escala MWconv, Perú/Bolivia y tiempo

In [ ]:
# a) Sensibilidad a Mc
mc_tab = {}
for mc in np.round(np.arange(4.0,5.01,0.1),1):
    mc_tab[mc] = [[int(((df.z==k)&(df.mag>=mc)).sum()), b_mle(df.mag[(df.z==k)&(df.mag>=mc)], mc)] for k in range(5)]
print(pd.DataFrame({m:[round(x[1],3) for x in v] for m,v in mc_tab.items() if m in (4.0,4.2,4.5)}, index=ZN))

# b) Escala: usar MWconv en lugar de mag
mw = [b_mle(df.MWconv[(df.z==k)&(df.MWconv>=MC)]) for k in range(5)]
print('\nb con MWconv:', np.round(mw,3), '| b con mag:', R.b.round(3).values)

# c) Norte Grande sin Perú/Bolivia
ng = d[(d.z==0)&(d.lat<-18.35)]
pb = dict(n=int(len(ng)), b=float(b_mle(ng.mag)), n_fuera=int(((d.z==0)&(d.lat>=-18.35)).sum()))
print('\nNorte Grande solo Chile: n=%d, b=%.3f  (con Perú/Bolivia: n=%d, b=%.3f)' % (pb['n'],pb['b'],R.n[0],R.b[0]))

# d) b por período
per=[(1900,1989),(1990,2004),(2005,2015),(2016,2025)]
per_b, per_n = [], []
for k in range(5):
    r1,r2=[],[]
    for a,bb in per:
        m=d.mag[(d.z==k)&(d.anio>=a)&(d.anio<=bb)]; r2.append(len(m)); r1.append(float(b_mle(m)) if len(m)>20 else None)
    per_b.append(r1); per_n.append(r2)
print(pd.DataFrame(per_b, index=ZN, columns=['1900-89','1990-2004','2005-15','2016-25']).round(3))

## 8. ¿Cuándo ayuda la contracción? Submuestreo
Se toman $n$ eventos sin reemplazo por zona (R = 500), se estima con MLE, EB y James–Stein, y se mide el error cuadrático contra el $b$ de la muestra completa. Razón < 1 = mejora sobre MLE.

In [ ]:
def james_stein(b, s2):
    b=np.asarray(b); mu=b.mean(); S=((b-mu)**2).sum()
    c=max(0, 1-(len(b)-3)*np.mean(s2)/S); return mu + c*(b-mu)

mags=[d.mag[d.z==k].values for k in range(5)]; btrue=R.b.values; sim=[]
for n in [20,30,50,100,200,350]:
    e_mle=e_eb=e_js=0
    for _ in range(R_SUB):
        b=np.array([b_mle(rng.choice(m,n,replace=False)) for m in mags]); s2=b**2/n
        e_mle+=((b-btrue)**2).sum(); e_eb+=((eb_dl(b,s2)['theta']-btrue)**2).sum(); e_js+=((james_stein(b,s2)-btrue)**2).sum()
    sim.append([n, round(e_eb/e_mle,3), round(e_js/e_mle,3)])
S = pd.DataFrame(sim, columns=['n','EB/MLE','JS/MLE']); S['reduccion_MSE_EB_%']=((1-S['EB/MLE'])*100).round(0); S

In [ ]:
plt.figure(figsize=(7,4)); plt.semilogx(S.n,S['EB/MLE'],'o-',label='Bayes Empírico'); plt.semilogx(S.n,S['JS/MLE'],'s--',label='James–Stein')
plt.axhline(1,c='gray',ls=':'); plt.xlabel('eventos por zona'); plt.ylabel('ECM relativo a MLE'); plt.legend(); plt.show()

## 9. Conclusiones
1. **Diferencias zonales:** b va de ≈0,50 (Austral) a ≈0,94 (Central); la heterogeneidad es muy significativa y persiste al cambiar $M_c$.
2. **MLE ≈ Bayes Empírico** con zonas grandes y heterogéneas (contracción 0,4–5,4 %).
3. **Con pocos eventos por zona, la contracción reduce el ECM**; con la muestra completa no hay mejora práctica.
4. **Límites:** escalas mixtas (MWconv cambia b), un solo $M_c$, dependencia temporal (SE por bloques mayor), Zona Austral con ajuste exponencial débil, ≈11 % de eventos fuera de Chile en Norte Grande y solo 5 zonas (poca información para estimar τ).

## 10. Exportar resultados al panel interactivo
Esta celda guarda `datos_panel.json` y genera `panel.html` (página única, sin servidor) a partir de `panel_template.html`. Para modificar el diseño, edita la plantilla; para actualizar números, vuelve a ejecutar el notebook.

In [ ]:
ms_gr = np.round(np.arange(MC,7.65,0.1),1)
h2,_ = np.histogram(df.mag, bins=np.arange(3.65,8.05,DM))
g = df.assign(gx=np.floor(df.lon).astype(int), gy=np.floor(df.lat).astype(int)).groupby(['gx','gy','z']).size().reset_index(name='n')
Z=[dict(n=int(R.n[k]), b=round(R.b[k],4), se=round(R.se[k],4), lo=round(R.ic_lo[k],3), hi=round(R.ic_hi[k],3),
        sb=round(R.se_bloques[k],4), sj=round(R.se_jackknife[k],4), ks=round(R.KS_D[k],3)) for k in range(5)]
datos = dict(zn=ZN, Z=Z,
  mc={str(m):[[v[0], round(v[1],3)] for v in vv] for m,vv in mc_tab.items()},
  gr=[[int((d.mag[d.z==k]>=m-1e-9).sum()) for m in ms_gr] for k in range(5)],
  per=per_b, perN=per_n,
  yc=[int(yc.get(y,0)) for y in range(1960,2026)], hist=[int(x) for x in h2],
  map=[[int(a),int(b),int(c),int(n)] for a,b,c,n in g.values],
  sim=sim, mw=[round(x,3) for x in mw], pb=pb,
  qb=dict(Q=round(Eb['Q'],1), p=float(Eb['p']), cmin=round(100*Eb['B'].min(),1), cmax=round(100*Eb['B'].max(),1)),
  q=dict(Q=round(E['Q'],1), p=float(E['p']), tau=round(E['tau'],3), mu=round(E['mu'],3)))
json.dump(datos, open('datos_panel.json','w'), separators=(',',':'))
import os
if os.path.exists('panel_template.html'):
    open('panel.html','w').write(open('panel_template.html',encoding='utf-8').read().replace('__DATA__', json.dumps(datos, separators=(',',':'))))
    print('Listo: panel.html generado (ábrelo en el navegador).')
else:
    print('datos_panel.json guardado. Sube panel_template.html para generar panel.html.')